# CropGuard — YOLOv8s Fine-Tune (Field Images)
**Run after:** `kaggle_train_potato_yolo.ipynb` (base training on PlantVillage).  
**Starting weights:** `best.pt` from the base run (mounted as `cropguard-weights` dataset).  
**New data:** `cropguard-field-potato` dataset (422 cleaned, remapped field images).  
**Do NOT** run cells out of order.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q ultralytics onnxslim wandb

In [ ]:
# Cell 2 — Clone repo (branch pinned at clone time — do not change)
!git clone --branch ml-1 https://github.com/aliviahossain/Crop_detection_management.git repo
%cd repo

In [ ]:
# Cell 3 — W&B login (optional — training continues if secret missing)
try:
    import wandb
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    wandb.login(key=secrets.get_secret("WANDB_API_KEY"))  # case-sensitive
    print("wandb ready")
except Exception as e:
    print(f"wandb skipped: {e}")
    print("Training will continue without W&B logging.")

In [ ]:
# Cell 4 — Verify best.pt is mounted and non-zero before doing anything
import os, sys

weights_path = "/kaggle/input/datasets/omsinghlodhi/cropguard-weights/best.pt"

if not os.path.exists(weights_path):
    sys.exit(f"ERROR: best.pt not found at {weights_path}. Did you mount cropguard-weights?")

size_mb = os.path.getsize(weights_path) / (1024 * 1024)
if size_mb < 1.0:
    sys.exit(f"ERROR: best.pt is only {size_mb:.2f} MB — file is likely corrupt.")

print(f"OK: best.pt found. Size: {size_mb:.2f} MB")

In [ ]:
# Cell 5 — Build base YOLO split from PlantVillage
# Flags must match the original training run exactly
!python ml/prepare_dataset.py \
    --plantvillage /kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color \
    --out /kaggle/working/datasets/potato_yolo \
    --cap-train 400 \
    --oversample-min \
    --clean

In [ ]:
# Cell 6a — Merge field images into the YOLO split
!python ml/merge_field.py \
    --field-dir /kaggle/input/datasets/omsinghlodhi/cropguard-field-potato/field_flat \
    --out /kaggle/working/datasets/potato_yolo \
    --seed 42 \
    --ratios 0.8 0.1 0.1

In [ ]:
# Cell 6b — Verify merge: check image counts per split
# Expected: train ~700+, valid ~100+, test ~80+
!echo -n "Train images: " && ls /kaggle/working/datasets/potato_yolo/images/train | wc -l
!echo -n "Val images:   " && ls /kaggle/working/datasets/potato_yolo/images/val | wc -l
!echo -n "Test images:  " && ls /kaggle/working/datasets/potato_yolo/images/test | wc -l

In [ ]:
# Cell 7 — Fine-tune from best.pt
# Key differences from base training:
#   --model  : best.pt not yolov8s.pt
#   --lr0    : 0.001 (10x lower than default to avoid catastrophic forgetting)
#   --epochs : 50 (fine-tuning converges faster than training from scratch)
!python ml/train_yolo.py \
    --model /kaggle/input/datasets/omsinghlodhi/cropguard-weights/best.pt \
    --data /kaggle/working/datasets/potato_yolo/data.yaml \
    --epochs 50 \
    --imgsz 640 \
    --batch 16 \
    --lr0 0.001 \
    --project /kaggle/working/runs \
    --name yolov8s_finetune_run1

In [ ]:
# Cell 8 — Copy weights and metrics to output directory
# Download these before your Kaggle session expires
!mkdir -p /kaggle/working/ml_weights
!cp /kaggle/working/runs/yolov8s_finetune_run1/weights/best.pt  /kaggle/working/ml_weights/
!cp /kaggle/working/runs/yolov8s_finetune_run1/weights/best.onnx /kaggle/working/ml_weights/ 2>/dev/null || echo 'ONNX not found — export may have failed'
!cp /kaggle/working/runs/yolov8s_finetune_run1/metrics.json      /kaggle/working/ml_weights/ 2>/dev/null || true
!cp /kaggle/working/runs/yolov8s_finetune_run1/results.png       /kaggle/working/ml_weights/ 2>/dev/null || true

!echo "--- Artifacts in /kaggle/working/ml_weights/ ---"
!ls -lh /kaggle/working/ml_weights/

In [ ]:
# Cell 10 — Re-tune thresholds on the fine-tuned model
# REQUIRED: confidence distributions shift after fine-tuning.
# Old thresholds.json from the base run is now stale — do not reuse it.
!python ml/tune_thresholds.py \
    --weights /kaggle/working/runs/yolov8s_finetune_run1/weights/best.pt \
    --data /kaggle/working/datasets/potato_yolo/data.yaml

# Copy new thresholds to output so they travel with the weights
!cp /kaggle/working/ml_weights/thresholds.json /kaggle/working/ml_weights/ 2>/dev/null || true

!echo "--- Final artifacts (download all of these) ---"
!ls -lh /kaggle/working/ml_weights/

In [ ]:
# Cell 10 — Re-tune thresholds on the fine-tuned model
# REQUIRED: confidence distributions shift after fine-tuning.
# Old thresholds.json from the base run is now stale — do not reuse it.
!python ml/tune_thresholds.py \
    --weights /kaggle/working/ml_weights/best.pt \
    --data /kaggle/working/datasets/potato_yolo/data.yaml

# Copy new thresholds to output so they travel with the weights
!cp /kaggle/working/ml_weights/thresholds.json /kaggle/working/ml_weights/ 2>/dev/null || true

!echo "--- Final artifacts (download all of these) ---"
!ls -lh /kaggle/working/ml_weights/